In [ ]:
import numpy as np
import pandas as pd
import re
from tsvpylab import meshmanager
from tsvpylab.meshmanager import gType
from tsvpylab import stagemanager
import os
import pyvista as pv
#root = r'O:\alves\AI\2025_Cogging_Internship\Cogging.tsv'
root = r'C:\Users\tcnguyen\FORGE\Simple_Cogging.tsv\Data_Old'
import h5py
from sklearn.model_selection import train_test_split

os.chdir(root)
os.listdir()

['HDF5_Detailed_Components.csv', 'Image', 'Sub_Datasets']

In [4]:

DST_FILE = "./Sub_Datasets/Dataset_Temp.h5" 

with h5py.File(DST_FILE, 'r') as f_dst:
    dst_val = f_dst['data'][:,:,:,:] 
    print(f"Destination Full Features: {dst_val}")
dst_val.shape


Destination Full Features: [[[[[-3.35851868e+02  5.98588943e+00 -1.14998001e+02  9.00000000e+02]
    [-3.35851868e+02  5.98588943e+00 -1.14998001e+02  9.00000000e+02]
    [-3.35851868e+02  5.98588943e+00 -1.14998001e+02  9.00000000e+02]
    ...
    [-3.42572632e+02  6.07110405e+00 -1.23114449e+02  8.98396057e+02]
    [-3.43557587e+02  6.06234932e+00 -1.24106049e+02  8.98201538e+02]
    [-3.44567780e+02  6.05436039e+00 -1.25086555e+02  8.98008362e+02]]

   [[-2.77534809e+01  1.32953014e+01 -1.00153786e+02  9.00000000e+02]
    [-2.77534809e+01  1.32953014e+01 -1.00153786e+02  9.00000000e+02]
    [-2.77534809e+01  1.32953014e+01 -1.00153786e+02  9.00000000e+02]
    ...
    [-2.76035919e+01  1.34201174e+01 -1.00420868e+02  8.99544983e+02]
    [-2.76038933e+01  1.34387150e+01 -1.00436859e+02  8.99495422e+02]
    [-2.76093960e+01  1.34613104e+01 -1.00454842e+02  8.99457581e+02]]

   [[-3.13568695e+02  1.67043095e+01  6.83639221e+01  9.00000000e+02]
    [-3.13568695e+02  1.67043095e+01  6.836

(500, 40, 300, 21, 4)

In [ ]:

X_train, X_val = train_test_split(
    X,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(X_train.shape)  # (16000, 300, 3)
print(X_val.shape)    # (4000, 300, 3)


In [19]:
data = dst_val.reshape(500*40,300,21,4)
np.shape(data)

(20000, 300, 21, 4)

In [21]:
data_to_save = data[::50,:,15,:-1].copy()
data_to_save

array([[[-339.78336  ,    6.0133715, -120.03196  ],
        [ -27.6363   ,   13.374693 , -100.37636  ],
        [-317.5556   ,   16.712324 ,   63.343784 ],
        ...,
        [-423.92416  ,   92.4837   , -118.90149  ],
        [-313.33316  ,  -42.7359   , -120.02995  ],
        [-423.9682   ,  -99.87531  ,  -22.906887 ]],

       [[-115.93714  ,   26.16017  , -119.98821  ],
        [-119.49195  ,  -78.08395  , -121.70849  ],
        [-108.95161  ,   23.668243 ,   57.436268 ],
        ...,
        [ -46.03121  ,  106.54307  ,  -21.900198 ],
        [ -39.055664 , -106.88523  ,  -45.520035 ],
        [-108.0951   , -102.1754   ,   34.406113 ]],

       [[ -25.800852 ,   20.346706 ,  -40.75957  ],
        [ -26.159008 ,    6.528274 ,  -41.274708 ],
        [ -24.40326  ,  -14.804972 ,  -42.4204   ],
        ...,
        [ -68.12305  , -105.15209  , -105.39593  ],
        [ -54.13795  , -108.59912  ,  -34.88088  ],
        [ -58.377254 ,   88.75199  ,   14.722926 ]],

       ...,

      

In [22]:
file_path = "C:/Users/tcnguyen/Downloads/Python code/Point-MAE-main/test_dataset.h5"  # Windows

with h5py.File(file_path, "w") as f:
    f.create_dataset("data", data=data_to_save)

In [ ]:
def normalize_pc(pc):
    """Normalize a single point cloud to a unit sphere centered at origin."""
    # 1. Center the points (Mean Centering)
    centroid = np.mean(pc, axis=0)
    pc = pc - centroid
    
    # 2. Scale the points (Unit Sphere Scaling)
    dist = np.max(np.sqrt(np.sum(pc**2, axis=1)))
    pc = pc / dist
    return pc

# Load your raw test data
with h5py.File('C:/Users/tcnguyen/Downloads/Python code/Point-MAE-main/test_dataset.h5', 'r') as f:
    # Adjust 'data' to whatever your key name is in the H5 file
    raw_data = np.array(f['data']) 

print(f"Original Range: {np.min(raw_data)} to {np.max(raw_data)}")

# Normalize all 20,000 samples
norm_data = np.zeros_like(raw_data)
for i in range(len(raw_data)):
    norm_data[i] = normalize_pc(raw_data[i])

print(f"Normalized Range: {np.min(norm_data)} to {np.max(norm_data)}")

# Save to the data folder
with h5py.File('C:/Users/tcnguyen/Downloads/Python code/Point-MAE-main/data/MyDataset/test_dataset_normalized.h5', 'w') as f:
    f.create_dataset('data', data=norm_data)
    # Also save dummy labels or IDs if your loader expects them
    f.create_dataset('label', data=np.zeros(len(norm_data))) 

print("Success: Normalized file saved at data/MyDataset/test_dataset_normalized.h5")

Original Range: -437.4673767089844 to 121.17715454101562
Normalized Range: -0.8814747333526611 to 0.8993148803710938
Success: Normalized file saved at data/MyDataset/test_dataset_normalized.h5


In [2]:
import numpy as np
import h5py

def normalize_pc(pc):
    """Normalize a single point cloud and return the parameters used."""
    # 1. Center the points (Mean Centering)
    centroid = np.mean(pc, axis=0) # Stock this [x, y, z]
    centered_pc = pc - centroid
    
    # 2. Scale the points (Unit Sphere Scaling)
    # This distance is the 'scale' factor
    max_dist = np.max(np.sqrt(np.sum(centered_pc**2, axis=1)))
    
    # Avoid division by zero for empty/single point clouds
    if max_dist == 0:
        max_dist = 1.0
        
    normalized_pc = centered_pc / max_dist
    
    return normalized_pc, centroid, max_dist

# Load your raw test data
raw_path = 'C:/Users/tcnguyen/Downloads/Python code/Point-MAE-main/test_dataset.h5'
with h5py.File(raw_path, 'r') as f:
    raw_data = np.array(f['data']) 

print(f"Original Range: {np.min(raw_data)} to {np.max(raw_data)}")

# Prepare arrays to hold normalized data and parameters
num_samples = len(raw_data)
norm_data = np.zeros_like(raw_data)
centroids = np.zeros((num_samples, 3))   # Store X, Y, Z of centers
scales = np.zeros((num_samples,))         # Store the max_dist

# Process all samples
for i in range(num_samples):
    norm_pc, center, scale = normalize_pc(raw_data[i])
    norm_data[i] = norm_pc
    centroids[i] = center
    scales[i] = scale

print(f"Normalized Range: {np.min(norm_data)} to {np.max(norm_data)}")

# Save to the data folder with extra metadata
save_path = 'C:/Users/tcnguyen/Downloads/Python code/Point-MAE-main/data/MyDataset/test_dataset_normalized.h5'
with h5py.File(save_path, 'w') as f:
    # Main normalized data
    f.create_dataset('data', data=norm_data)
    
    # Metadata for reversing normalization
    f.create_dataset('centroid', data=centroids)
    f.create_dataset('max_dist', data=scales)
    
    # Dummy labels
    f.create_dataset('label', data=np.zeros(num_samples)) 

print(f"Success: Normalized file and recovery params saved at: {save_path}")

Original Range: -437.4673767089844 to 121.17715454101562
Normalized Range: -0.8814747333526611 to 0.8993148803710938
Success: Normalized file and recovery params saved at: C:/Users/tcnguyen/Downloads/Python code/Point-MAE-main/data/MyDataset/test_dataset_normalized.h5
